# Transaction Foundation Model on Ray — Part 2: Load & explore the data w/ Ray Data

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~15 min (most of it the one-time dataset download + normalize)


---

In Part 1, we set up the cluster. Here we load the real benchmark dataset.

In [9]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

2026-06-24 22:59:32,215	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.188.86:6379...
2026-06-24 22:59:32,216	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


Python version:,3.12.13
Ray version:,2.55.1
Dashboard:,http://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com


## IBM TabFormer Dataset

We use the **IBM TabFormer** dataset (Padhi et al., ICASSP 2021) — the de-facto public benchmark for transaction foundation models. It has **24.4M** card transactions, ~6.1k cards, 1991–2020, ~0.12% transaction-level fraud. NVIDIA's transaction-FM blueprint and FATA-Trans both evaluate on it, so our downstream numbers are comparable.


We load the 24.4M rows as a Ray Data Pipeline that streams across the cluster. Something like Pandas might OOM on this relatively small dataset, but Ray Data, allows us to load datasets of any scale by distributing it through a scalable cluster.

General approach:
1. **Read: `read_csv`** — read the raw file in parallel
2. **Normalize: `map_batches(normalize_batch)`** — parse each batch into our canonical schema (things like - `"$57.20"` → `57.20`, MCC → category, modal home state — lives in `src/tabformer.py`; it's dataset-specific and a bit in the weeds).
3. **Aggregate: `groupby("card_id").map_groups(card_statics)`** — a distributed aggregation for the per-card fields. These "static" fields are broadcast back into each transation row.
4. **Write out: `write_parquet`** — a streamed write to sharded Parquet.

Some dataset-specific callbacks are imported `src/tabformer.py` and we compose the pipeline here. See `prepare_tabformer()` in that module for the identical pipeline wrapped for headless (scripts/job) runs.

### A note on `SCALE`

`SCALE` is one knob that picks a preset of everything that grows with the run — here, how many cards to sample; in later notebooks, the sequence length, model size, and number of GPU workers. The three presets (`smoke` / `small` / `full`) live in `configs/`. We use **`smoke`**: tiny and CPU-only, so this notebook runs in minutes. It's the *same code, change config* idea — flip one variable to go from a laptop-sized run to a full multi-GPU one. Outputs are written under a per-scale path (`.../raw/smoke/…`) so different scales don't overwrite each other.

In [ ]:
from src.paths import artifact_paths, get_demo_base_dir, write_splits_meta
from src.scale_config import load_scale
from src.tabformer import (
    ensure_download, normalize_batch, card_statics,
    sample_cards, attach_statics, tabformer_csv_convert_options,
)

SCALE = "smoke"            # which configs/<SCALE>.yaml preset to run; smoke = tiny, CPU-only
cfg = load_scale(SCALE)    # the configs/smoke.yaml settings (num_cards here; model/training later)
paths = artifact_paths(get_demo_base_dir(), SCALE)   # outputs namespaced per scale: .../<stage>/smoke/

# Build the canonical dataset once; re-runs reuse the cached Parquet shards.
if not (os.path.exists(paths["raw"]) and os.path.exists(paths["splits"])):
    csv_path = ensure_download(paths["source"])   # one-time ~266MB download + extract

    # Read 24.4M rows and normalize to our schema — streamed across the cluster.
    ds = ray.data.read_csv(
        csv_path, convert_options=tabformer_csv_convert_options(),
    ).map_batches(normalize_batch, batch_format="pandas")
    ds = sample_cards(ds, cfg["data"]["num_cards"]).materialize()   # smoke -> 2,000 cards; cache the read (reused below)

    # Per-card STATIC fields via a distributed groupby, broadcast onto every txn.
    statics = ds.groupby("card_id").map_groups(card_statics, batch_format="pandas").to_pandas()
    ds = attach_statics(ds, statics).materialize()          # cache before the two consumers below

    ds.write_parquet(paths["raw"])                 # streamed write -> Parquet shards

    # Temporal 80/10/10 cutoffs every later stage reuses. Only the 2 columns the
    # split needs come to the driver — not the whole table.
    slim = ds.select_columns(["timestamp", "is_fraud"]).to_pandas()
    write_splits_meta(paths["splits"], slim["timestamp"].to_numpy(),
                      slim["is_fraud"].to_numpy(), source="tabformer", n_cards=len(statics))

In [ ]:
# Load the normalized transactions back, sorted into per-card time order.
df = pd.read_parquet(paths["raw"]).sort_values(["card_id", "timestamp"])
with open(paths["splits"]) as f:
    splits = json.load(f)

print(f"{len(df):,} transactions  /  {df['card_id'].nunique():,} cards  "
      f"/  {df['is_fraud'].mean()*100:.3f}% fraud")
df.head(5)

## The static / dynamic split — visible in the schema

This is the core modeling idea of the template. Each transaction has two kinds of field:

- **Static** (card-level): `issuer`, `bin_region`, `card_type`, `home_state` — they describe the *card* and never change within its sequence.
- **Dynamic** (per-transaction): `amount`, `merchant_id`, `merchant_category`, `mcc`, `hour`, `day_of_week` — they change every event.

The tokenizer (Part 3) embeds the static fields **once** and broadcasts them to every position, while each dynamic field gets its own embedding table and occupies **one** position per transaction. The check below proves the premise: count distinct values per card and the static fields collapse to 1.

In [ ]:
STATIC_COLS  = ["issuer", "bin_region", "card_type", "home_state"]
DYNAMIC_COLS = ["amount", "merchant_id", "merchant_category", "mcc", "hour", "day_of_week"]

# Max distinct values per card: statics collapse to 1, dynamics vary.
per_card_nunique = df.groupby("card_id")[STATIC_COLS + DYNAMIC_COLS].nunique().max()

print("kind      field                max distinct values per card")
print("-" * 56)
for c in STATIC_COLS:
    print(f"static    {c:<18}   {per_card_nunique[c]:>10,}")
for c in DYNAMIC_COLS:
    print(f"dynamic   {c:<18}   {per_card_nunique[c]:>10,}")

Note `issuer` and `bin_region` are constant (`UNKNOWN`) because TabFormer doesn't carry them — the loader derives `card_type` and `home_state` per card and leaves the rest as a documented `UNKNOWN` bucket. On richer real data these static fields carry real signal, and the split pays off more.

## One card is one sequence

The model consumes a card's transactions as a time-ordered sequence. Here is what one looks like — the dynamic fields the model will learn to predict (masked-feature modeling, Part 4).

In [ ]:
card = df["card_id"].iloc[0]
seq = df[df["card_id"] == card]
print(f"card {card}: {len(seq):,} transactions from "
      f"{seq['timestamp'].min().date()} to {seq['timestamp'].max().date()}")
seq[["timestamp", "amount", "merchant_category", "mcc",
     "hour", "day_of_week", "is_fraud"]].head(10)

## Dataset shape and the temporal split

In [ ]:
seq_lens = df.groupby("card_id").size()
print(f"transactions ............ {len(df):,}")
print(f"cards ................... {df['card_id'].nunique():,}")
print(f"txns per card ........... median {int(seq_lens.median()):,}  "
      f"(min {seq_lens.min():,}, max {seq_lens.max():,})")
print(f"date range .............. {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")
print(f"overall fraud rate ...... {df['is_fraud'].mean()*100:.3f}%  "
      f"({int(df['is_fraud'].sum()):,} fraudulent txns)")
print()
print("temporal split (from splits.json — the same cutoffs every stage uses):")
print(f"  train  <  {splits['train_end']}")
print(f"  val    <  {splits['val_end']}")
print(f"  test   >= {splits['val_end']}")

**Why split by time, not at random?** Fraud detection is a forecasting problem — you train on the past and score the future. A random split leaks future transactions of the same card into training and inflates the metrics. We train on the oldest 80%, early-stop on the next 10%, and report on the most recent 10% — the NVIDIA blueprint protocol.

## Four pictures, four modeling decisions

Each panel below motivates a specific choice you'll see in the next notebooks:

1. **Transactions per card** → how long a sequence window (`seq_len`) needs to be.
2. **Amount is heavy-tailed** → we **log-bucket** amounts into categorical tokens instead of feeding a raw scalar.
3. **Volume over time** → where the temporal train/val/test cutoffs fall.
4. **Bursty inter-transaction gaps** → why positions are **time-aware** (we embed the log-bucketed gap since the previous transaction, not just the ordinal slot).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) transactions per card -> motivates the sequence window length (seq_len).
ax = axes[0, 0]
ax.hist(seq_lens, bins=40, color="#4C78A8")
ax.set_title("Transactions per card (sequence length)")
ax.set_xlabel("transactions"); ax.set_ylabel("cards")

# (2) amount is heavy-tailed -> we log-bucket it in the tokenizer.
ax = axes[0, 1]
log_amt = np.log10(np.clip(np.abs(df["amount"].to_numpy()), 0.1, None))
ax.hist(log_amt, bins=60, color="#F58518")
ax.set_title("Transaction amount (log10 $) — heavy-tailed")
ax.set_xlabel("log10(amount)"); ax.set_ylabel("transactions")

# (3) volume over time with the temporal 80/10/10 split boundaries.
ax = axes[1, 0]
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size()
ax.plot(monthly.index.to_timestamp(), monthly.values, color="#54A24B")
for cut, lbl in [(splits["train_end"], "train|val"), (splits["val_end"], "val|test")]:
    ax.axvline(pd.Timestamp(cut), color="crimson", ls="--", lw=1)
    ax.text(pd.Timestamp(cut), ax.get_ylim()[1] * 0.92, lbl, rotation=90,
            va="top", ha="right", color="crimson", fontsize=8)
ax.set_title("Transaction volume over time + temporal split")
ax.set_xlabel("month"); ax.set_ylabel("transactions")

# (4) inter-transaction gaps -> motivates time-aware position embeddings.
ax = axes[1, 1]
gaps = df.groupby("card_id")["timestamp"].diff().dt.total_seconds() / 3600.0
gaps = gaps.dropna()
gaps = gaps[gaps > 0]
ax.hist(np.log10(gaps), bins=60, color="#B279A2")
ax.set_title("Hours between consecutive txns (log10) — bursty")
ax.set_xlabel("log10(hours since previous txn)"); ax.set_ylabel("transactions")

plt.tight_layout()
plt.show()

## Class imbalance — and why we don't report plain accuracy

At **~0.12%** fraud, a model that predicts "never fraud" is 99.88% accurate and useless. Two consequences drive the rest of the series:

- **Metric**: we report **PR-AUC** (average precision) alongside AUC-ROC. At this prevalence AUC-ROC saturates near 1.0 and hides differences; PR-AUC is the operationally meaningful number.
- **Sampling**: the tokenizer keeps **every** fraud but **downsamples normal** transactions for the eval set (compute), then attaches **importance weights** so the reported metrics still estimate performance at the *natural* prevalence. You'll see both in Part 6.

In [ ]:
fraud_rate = df["is_fraud"].mean()
print(f"normal transactions : {(1 - fraud_rate) * 100:.3f}%")
print(f"fraud transactions  : {fraud_rate * 100:.3f}%")
print(f"imbalance ratio     : ~1 fraud per {int(round(1 / fraud_rate)):,} transactions")

## Takeaways

- One card = one **time-ordered sequence** of transactions; fields split cleanly into **static** (card-level) and **dynamic** (per-transaction).
- Amounts are **heavy-tailed**, inter-event gaps are **bursty**, and fraud is **rare** — each shapes the tokenizer, the model, and the evaluation.
- The **temporal split** is fixed once, in `splits.json`, and reused by every downstream stage so there's no leakage and runs stay comparable.

---

## Next

**Part 3 — Tokenize with Ray Data**: turn these raw sequences into padded, per-field token windows using the static/dynamic split, as a stateless, embarrassingly parallel `map_groups` over the cluster.